In [ ]:
!pip install -q tensorflow keras

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras import layers, models, callbacks, optimizers, metrics
from tensorflow.keras.applications.xception import Xception, preprocess_input

print("TensorFlow Version:", tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/UADFV_processed"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "val")

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)

In [ ]:
IMAGE_SIZE = (299, 299)
BATCH_SIZE = 32
INITIAL_EPOCHS = 5
FINETUNE_EPOCHS = 5

BASE_LR = 3e-4
FINETUNE_LR = 1e-5

MODEL_PATH = os.path.join(DATASET_DIR, "trained_models", "xception_video_model.keras")
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR, label_mode="binary", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR, label_mode="binary", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

print("Classes:", train_ds.class_names)

In [ ]:
def extract_labels(dataset):
    labels = []
    for _, y in dataset.unbatch():
        labels.append(int(y.numpy().item()))
    return np.array(labels)

y_train = extract_labels(train_ds)
class_weights = class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(class_weights)}
class_weights

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1)
])

def preprocess_train(img, label):
    img = augmentation(img)
    img = preprocess_input(img)
    return img, label

def preprocess_val(img, label):
    return preprocess_input(img), label

train_ds = train_ds.map(preprocess_train).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_val).cache().prefetch(AUTOTUNE)

In [ ]:
base_model = Xception(weights="imagenet", include_top=False, input_shape=(299,299,3))
base_model.trainable = False

inputs = layers.Input(shape=(299,299,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

model = models.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=optimizers.Adam(BASE_LR),
    loss="binary_crossentropy",
    metrics=["accuracy", metrics.AUC(), metrics.Precision(), metrics.Recall()]
)

callbacks_list = [
    callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_auc", mode="max"),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.2)
]

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights, 
    callbacks=callbacks_list
)

In [ ]:
base_model.trainable = True

total_layers = len(base_model.layers)
freeze_until = int(total_layers * 0.6)

for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= freeze_until)

model.compile(
    optimizer=optimizers.Adam(FINETUNE_LR),
    loss="binary_crossentropy",
    metrics=["accuracy", metrics.AUC(), metrics.Precision(), metrics.Recall()]
)

In [ ]:
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_list
)

In [ ]:
y_true, y_pred, y_prob = [], [], []

for images, labels in val_ds.unbatch().batch(32):
    preds = model.predict(images).flatten()
    y_prob.extend(preds)
    y_pred.extend((preds > 0.5).astype(int))
    y_true.extend(labels.numpy().astype(int))

print("\nAUC:", roc_auc_score(y_true, y_prob))
print("\nClassification Report:\n", classification_report(y_true, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
model.save(MODEL_PATH)
print("Model saved at:", MODEL_PATH)